# TikTok TechJam AIGC detector — Colab smoke run

This notebook runs the real CIFAKE → DINOv2 → checkpoint → competition JSON path. It keeps the image dataset on Colab's fast temporary disk and checkpoints on Google Drive.

Before running: choose **Runtime → Change runtime type → T4 GPU**. The one-epoch run is an integration test, not a publishable benchmark. CIFAKE contains only 32×32 images from one synthetic generator.

## 1. Load the project

Use a GitHub URL after you push the repository. If `REPO_URL` is empty, Colab will ask you to upload `colab_bundle.zip`, which can be created from this repository.

In [ ]:
from pathlib import Path
import os
import subprocess
import zipfile

REPO_URL = ""  # Example: https://github.com/your-name/your-repo.git
BRANCH = "main"
PROJECT_DIR = Path("/content/tiktok-techjam")

if PROJECT_DIR.exists():
    print(f"Reusing existing project directory: {PROJECT_DIR}")
elif REPO_URL.strip():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
else:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one colab_bundle.zip file")
    archive_name, archive_bytes = next(iter(uploaded.items()))
    if not archive_name.lower().endswith(".zip"):
        raise RuntimeError("The uploaded source bundle must be a .zip file")
    archive_path = Path("/content") / archive_name
    archive_path.write_bytes(archive_bytes)
    PROJECT_DIR.mkdir(parents=True)
    with zipfile.ZipFile(archive_path) as bundle:
        destination = PROJECT_DIR.resolve()
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f"Unsafe archive member: {member.filename}")
        bundle.extractall(destination)

os.chdir(PROJECT_DIR)
required = ["train.py", "predict.py", "requirements.txt", "configs/smoke_cifake_dino.yaml"]
missing = [name for name in required if not (PROJECT_DIR / name).is_file()]
if missing:
    raise RuntimeError(f"Project source is incomplete; missing {missing}")
print("Project ready at", PROJECT_DIR)

## 2. Verify the GPU and install dependencies

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Choose Runtime → Change runtime type → T4 GPU, then reconnect."
    )
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory: %.1f GiB" % (torch.cuda.get_device_properties(0).total_memory / 2**30))
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Dependencies installed")

## 3. Mount Drive for persistent checkpoints

Only checkpoints/results go to Drive. The 120,000 small image files stay on `/content` for much faster reads.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/TikTokTechJam2026")
RUN_DIR = DRIVE_ROOT / "runs" / "smoke_cifake_dino"
RESULTS_DIR = DRIVE_ROOT / "results" / "smoke_cifake_dino"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent run directory:", RUN_DIR)

## 4. Download and validate CIFAKE

The public Kaggle archive is about 112 MB and extracts to about 469 MB. This cell is idempotent: it skips the download when all 120,000 images are already present in this runtime.

In [ ]:
DATA_ROOT = PROJECT_DIR / "data" / "cifake"
ARCHIVE = PROJECT_DIR / "data" / "raw" / "cifake.zip"
KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/birdy654/cifake-real-and-ai-generated-synthetic-images"
SUPPORTED = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

def image_count(root):
    return sum(1 for path in root.rglob("*") if path.is_file() and path.suffix.lower() in SUPPORTED)

count = image_count(DATA_ROOT) if DATA_ROOT.exists() else 0
if count != 120_000:
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["curl", "--fail", "--location", "--retry", "3", "--output", str(ARCHIVE), KAGGLE_URL],
        check=True,
    )
    with zipfile.ZipFile(ARCHIVE) as dataset_zip:
        bad_member = dataset_zip.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt CIFAKE archive member: {bad_member}")
        dataset_zip.extractall(DATA_ROOT)
    count = image_count(DATA_ROOT)

if count != 120_000:
    raise RuntimeError(f"Expected 120000 CIFAKE images, found {count}")
print(f"CIFAKE ready: {count} images at {DATA_ROOT}")

In [ ]:
subprocess.run(
    [sys.executable, "-m", "src.data.audit_cli", "splits", "--config", "configs/smoke_cifake_dino.yaml", "--check-files"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "src.data.audit_cli", "shortcut", "--config", "configs/smoke_cifake_dino.yaml", "--sample", "5000"],
    check=True,
)

## 5. Train the one-epoch DINOv2 smoke run

The checkpoint is written directly to Drive. If `best.pt` already exists, the cell will not overwrite it. Delete or rename that run directory only when you intentionally want a fresh run.

In [ ]:
import time

best_checkpoint = RUN_DIR / "best.pt"
last_checkpoint = RUN_DIR / "last.pt"
if best_checkpoint.exists():
    print("Existing completed checkpoint found; training skipped:", best_checkpoint)
else:
    command = [
        sys.executable, "train.py",
        "--config", "configs/smoke_cifake_dino.yaml",
        "--output-dir", str(RUN_DIR),
        "--device", "cuda",
    ]
    if last_checkpoint.exists():
        command.extend(["--resume", str(last_checkpoint)])
        print("Resuming from", last_checkpoint)
    started = time.perf_counter()
    subprocess.run(command, check=True)
    print("Training wall time: %.1f minutes" % ((time.perf_counter() - started) / 60))

if not best_checkpoint.is_file():
    raise RuntimeError("Training finished without best.pt")
print("Best checkpoint:", best_checkpoint)

In [ ]:
import json

summary_path = RUN_DIR / "training_summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
else:
    print("No training_summary.json found")

## 6. Exercise the submission inference path

This copies two real and two fake CIFAKE examples into a demo directory. The output file must contain only `image_path` and `pred`, where `pred` is P(AIGC).

In [ ]:
import shutil

DEMO_DIR = PROJECT_DIR / "data" / "demo_images"
DEMO_DIR.mkdir(parents=True, exist_ok=True)
real_examples = sorted((DATA_ROOT / "test" / "REAL").glob("*.jpg"))[:2]
fake_examples = sorted((DATA_ROOT / "test" / "FAKE").glob("*.jpg"))[:2]
if len(real_examples) != 2 or len(fake_examples) != 2:
    raise RuntimeError("Could not select CIFAKE demo examples")
for index, source in enumerate(real_examples):
    shutil.copy2(source, DEMO_DIR / f"real_{index}.jpg")
for index, source in enumerate(fake_examples):
    shutil.copy2(source, DEMO_DIR / f"fake_{index}.jpg")

predictions_path = RESULTS_DIR / "predictions.json"
diagnostics_path = RESULTS_DIR / "robustness.json"
subprocess.run(
    [
        sys.executable, "predict.py",
        "--input", str(DEMO_DIR),
        "--output", str(predictions_path),
        "--checkpoint", str(best_checkpoint),
        "--device", "cuda",
        "--path-format", "input-relative",
        "--diagnostics-output", str(diagnostics_path),
    ],
    check=True,
)
rows = json.loads(predictions_path.read_text())
assert all(set(row) == {"image_path", "pred"} for row in rows)
assert all(0.0 <= float(row["pred"]) <= 1.0 for row in rows)
print(json.dumps(rows, indent=2))
print("Saved predictions to", predictions_path)
print("Saved diagnostics to", diagnostics_path)

## 7. What success means

After every cell passes, Drive contains a self-describing `best.pt`, a training summary, exact competition JSON, and robustness diagnostics. That proves the real data/training/product integration. It does **not** establish final detector quality.

Next, replace the smoke experiment with the comparable CLIP, DINOv2 and I-JEPA baseline configurations. Keep all model selection on validation. Do not use the protected COCO val2017/DALL·E Advanced demonstration subset for training or model selection.

For longer runs, keep output directories on Drive and close the Colab runtime when finished so it stops consuming compute units.